In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv", parse_dates=['date'])
test = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv", parse_dates=['date'])
stores = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv", parse_dates=['date'])
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=['date'])
oil = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv", parse_dates=['date'])

In [ ]:
from sklearn.metrics import mean_squared_log_error
from lightgbm import LGBMRegressor, early_stopping

In [ ]:
print(train.head(), train.shape)
print(test.head(), test.shape)

In [ ]:
oil = oil.sort_values("date")
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate()
oil["dcoilwtico"] = oil["dcoilwtico"].fillna(method="bfill")

hol = holidays.copy()
if "transferred" in hol.columns:
    hol = hol[~hol["transferred"].fillna(False)]
hol = hol[["date", "type", "locale"]].rename(columns={"type":"holiday_type"})
hol["is_holiday"] = 1

trans = transactions.groupby(["store_nbr","date"], as_index=False)["transactions"].sum()

for df in (train, test):
    df.merge(stores, on="store_nbr", how="left", copy=False)
    
train = train.merge(stores, on="store_nbr", how="left")
test  = test.merge(stores, on="store_nbr", how="left")

train = train.merge(oil, on="date", how="left")
test  = test.merge(oil, on="date", how="left")

train = train.merge(trans, on=["store_nbr","date"], how="left")
test  = test.merge(trans, on=["store_nbr","date"], how="left")

train = train.merge(hol, on="date", how="left")
test  = test.merge(hol, on="date", how="left")

train["is_holiday"] = train["is_holiday"].fillna(0).astype(int)
test["is_holiday"]  = test["is_holiday"].fillna(0).astype(int)

if "onpromotion" in train.columns:
    train["onpromotion"] = train["onpromotion"].fillna(0).astype(float)
if "onpromotion" in test.columns:
    test["onpromotion"] = test["onpromotion"].fillna(0).astype(float)

train["transactions"] = train["transactions"].fillna(0)
test["transactions"]  = test["transactions"].fillna(0)

print("After merges:", train.shape, test.shape)

In [ ]:
def add_time_parts(df):
    df["day"] = df["date"].dt.day
    df["dow"] = df["date"].dt.dayofweek  # 0=Mon
    df["week"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    return df

train = add_time_parts(train)
test  = add_time_parts(test)

In [ ]:
# Combine to build sequential features safely
target_col = "sales"
id_cols = ["id"] if "id" in test.columns else []

train["is_train"] = 1
test["is_train"] = 0
test[target_col] = np.nan  # placeholder for combined structure

cols_order = train.columns  # to keep a consistent column set

all_df = pd.concat([train, test[train.columns]], ignore_index=True).sort_values(["store_nbr","family","date"])

# Core group for sales history
grp_keys = ["store_nbr","family"]

def add_sales_lags(df, lags=(1,7,14,28)):
    g = df.groupby(grp_keys, sort=False)
    for L in lags:
        df[f"lag_{L}"] = g[target_col].shift(L)
    return df

def add_sales_rolls(df, windows=(7,14,28)):
    g = df.groupby(grp_keys, sort=False)[target_col]
    for W in windows:
        df[f"roll_mean_{W}"] = g.shift(1).rolling(W, min_periods=W).mean()
        df[f"roll_std_{W}"]  = g.shift(1).rolling(W, min_periods=W).std()
    return df

def add_txn_rolls(df, windows=(7,14,28)):
    g = df.groupby(["store_nbr"], sort=False)["transactions"]
    for W in windows:
        df[f"txn_mean_{W}"] = g.shift(1).rolling(W, min_periods=W).mean()
    return df

def add_oil_rolls(df, windows=(7,14,28)):
    s = df.sort_values("date")["dcoilwtico"]
    # Global rolling on date (no grouping)
    for W in windows:
        df[f"oil_mean_{W}"] = s.shift(1).rolling(W, min_periods=W).mean().values
    return df

all_df = add_sales_lags(all_df)
all_df = add_sales_rolls(all_df)
all_df = add_txn_rolls(all_df)
all_df = add_oil_rolls(all_df)

# After creating lags, we’ll split back
train_fe = all_df[all_df["is_train"] == 1].copy()
test_fe  = all_df[all_df["is_train"] == 0].copy()

# Drop rows in train that don't have full lag/rolling history yet
need_cols = [c for c in train_fe.columns if c.startswith(("lag_","roll_","txn_mean_","oil_mean_"))]
train_fe = train_fe.dropna(subset=need_cols)

# Clean up helpers
for df in (train_fe, test_fe):
    df.drop(columns=["is_train"], inplace=True)

In [ ]:
# Identify objects to encode
cat_obj_cols = (train_fe.select_dtypes(include="object").columns
                .tolist())

# Common trick: fit categories on the union, then transform both
both = pd.concat([train_fe[cat_obj_cols], test_fe[cat_obj_cols]], axis=0)
for col in cat_obj_cols:
    both[col] = both[col].astype("category")
cats = {col: both[col].cat.categories for col in cat_obj_cols}

def apply_cats(df, cats):
    for col, cat in cats.items():
        df[col] = pd.Categorical(df[col], categories=cat).codes
    return df

train_fe = apply_cats(train_fe, cats)
test_fe  = apply_cats(test_fe, cats)

In [ ]:
drop_cols = ["date", target_col] + (["id"] if "id" in train_fe.columns else [])
features = [c for c in train_fe.columns if c not in drop_cols]

X = train_fe[features]
y = train_fe[target_col]
X_test = test_fe[features]

X.shape, X_test.shape

In [ ]:
# Choose last 15 days of train as validation (adjust N if your sample is shorter)
N_VALID_DAYS = 15
cutoff_date = train_fe["date"].max() - pd.Timedelta(days=N_VALID_DAYS)

train_mask = train_fe["date"] <= cutoff_date
val_mask   = train_fe["date"] >  cutoff_date

X_tr, y_tr = X[train_mask], y[train_mask]
X_va, y_va = X[val_mask],   y[val_mask]

print("Train span:", train_fe.loc[train_mask, "date"].min(), "->", train_fe.loc[train_mask, "date"].max())
print("Valid span:", train_fe.loc[val_mask,   "date"].min(), "->", train_fe.loc[val_mask,   "date"].max())
print(X_tr.shape, X_va.shape)

In [ ]:
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

In [ ]:
SEED = 42
params = dict(
    objective="regression",
    learning_rate=0.05,
    num_leaves=63,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=SEED,
    n_estimators=5000  # large; early stopping will cap it
)

model = LGBMRegressor(**params)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="rmse",
    callbacks=[early_stopping(stopping_rounds=200)]
)

va_pred = model.predict(X_va)
print("Validation RMSLE:", rmsle(y_va, va_pred))

In [ ]:
best_n = getattr(model, "best_iteration_", params["n_estimators"])
final_model = LGBMRegressor(**{**params, "n_estimators": best_n})
final_model.fit(X, y)

test_pred = final_model.predict(X_test)
test_pred = np.maximum(test_pred, 0)

submission = pd.DataFrame({
    "id": test_fe["id"].values if "id" in test_fe.columns else test["id"].values,
    "sales": test_pred
}).sort_values("id")

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
submission.head(), submission.shape